# P3.2 – Model Pruning

Pruning removes parts of the model that contribute least to the output, reducing size and
inference cost.  We implement and compare two complementary strategies:

| Strategy | Type | What is removed | Size reduction |
|---|---|---|---|
| Attention head pruning | Width | Least-important heads within each layer | Moderate |
| Layer dropping | Depth | Entire encoder layers | Large |

**Importance scoring for attention heads** follows the Taylor-expansion proxy:
importance ≈ |weight × gradient|, averaged over a calibration set.

**Layer importance** is estimated by the average activation norm: layers with small norms
contribute least to the final representation.

> **Prerequisites:** Run `P3_1_Baseline_Evaluation.ipynb` first to generate
> `baseline_results.json` and `data_splits.pkl`.

In [ ]:
!pip install -q transformers datasets scikit-learn torch matplotlib seaborn

In [ ]:
import os
import copy
import json
import time
import shutil
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# ── Colab + VSCode path setup ─────────────────────────────────────────────────
BASE_DIR = '/content/P3-CustomFinancialLLM/'
os.makedirs(BASE_DIR, exist_ok=True)
print('Working dir:', BASE_DIR)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1 – Load model, data splits, and baseline results

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(os.path.join(BASE_DIR, 'data_splits.pkl'), 'rb') as f:
    splits = pickle.load(f)

train_texts  = splits['train_texts']
train_labels = splits['train_labels']
test_texts   = splits['test_texts']
test_labels  = splits['test_labels']
id2label     = splits['id2label']

with open(os.path.join(BASE_DIR, 'baseline_results.json')) as f:
    baseline = json.load(f)

print('Baseline accuracy : {:.2f}%'.format(baseline['accuracy'] * 100))
print('Baseline F1       : {:.4f}'.format(baseline['f1_weighted']))
print('Baseline params   : {}M'.format(baseline['params_M']))
print('Baseline size     : {}MB'.format(baseline['size_mb']))

In [ ]:
# Shared benchmark utility (mirrors Notebook 1)
def get_model_size_mb(model, tmp_dir='/tmp/p3_prune_size'):
    if os.path.exists(tmp_dir): shutil.rmtree(tmp_dir)
    model.save_pretrained(tmp_dir)
    total = sum(
        os.path.getsize(os.path.join(tmp_dir, fn))
        for fn in os.listdir(tmp_dir)
        if os.path.isfile(os.path.join(tmp_dir, fn))
    )
    shutil.rmtree(tmp_dir)
    return total / (1024 ** 2)


def benchmark_model(model, tokenizer, texts, labels, batch_size=16, label='model'):
    model.eval()
    all_preds = []
    t_start = time.time()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch  = texts[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True,
                               truncation=True, max_length=128)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            preds  = model(**inputs).logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
    elapsed = time.time() - t_start
    return {
        'label':         label,
        'accuracy':      round(accuracy_score(labels, all_preds), 4),
        'f1_weighted':   round(f1_score(labels, all_preds, average='weighted'), 4),
        'params_M':      round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
        'size_mb':       round(get_model_size_mb(model), 2),
        'ms_per_sample': round(elapsed * 1000 / len(texts), 3),
        'predictions':   all_preds,
    }

## 2 – Calibration set

Importance scores are computed on a small calibration set (not the test set).
Using 10–15% of the training data is sufficient.

In [ ]:
CAL_SIZE = min(256, len(train_texts))
cal_texts  = train_texts[:CAL_SIZE]
cal_labels = train_labels[:CAL_SIZE]


def get_calibration_batches(texts, labels, tokenizer, batch_size=16):
    batches = []
    for i in range(0, len(texts), batch_size):
        batch_t = texts[i:i + batch_size]
        batch_l = labels[i:i + batch_size]
        inputs  = tokenizer(batch_t, return_tensors='pt', padding=True,
                            truncation=True, max_length=128)
        inputs  = {k: v.to(DEVICE) for k, v in inputs.items()}
        inputs['labels'] = torch.tensor(batch_l, dtype=torch.long).to(DEVICE)
        batches.append(inputs)
    return batches


print('Calibration set size:', CAL_SIZE)

## 3 – Strategy A: Attention Head Pruning (Width)

### 3a – Compute head importance scores

We estimate each head's importance using the Taylor-expansion proxy:

$$I_h \approx \left| \mathbf{W}_h \cdot \nabla_{\mathbf{W}_h} \mathcal{L} \right|$$

Heads with the smallest score are pruned first.

In [ ]:
def compute_head_importance(model, calibration_batches):
    cfg = model.config
    n_layers = cfg.num_hidden_layers
    n_heads  = cfg.num_attention_heads
    head_dim = cfg.hidden_size // n_heads

    importance = torch.zeros(n_layers, n_heads)

    model.train()  # enable gradients
    for batch in calibration_batches:
        model.zero_grad()
        outputs = model(**batch)
        outputs.loss.backward()

        for layer_idx, layer in enumerate(model.bert.encoder.layer):
            attn = layer.attention.self
            # Query weight gradient as proxy
            grad = attn.query.weight.grad   # (hidden, hidden)
            w    = attn.query.weight.data
            # Reshape to (n_heads, head_dim, hidden) and take mean magnitude
            g_heads = grad.view(n_heads, head_dim, -1)
            w_heads = w.view(n_heads, head_dim, -1)
            imp = (w_heads * g_heads).abs().mean(dim=(1, 2))
            importance[layer_idx] += imp.detach().cpu()

    model.eval()
    importance /= len(calibration_batches)
    return importance


# Load a fresh model for pruning (don't modify the baseline)
model_head = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)

cal_batches = get_calibration_batches(cal_texts, cal_labels, tokenizer)

print('Computing head importance scores ...')
head_importance = compute_head_importance(model_head, cal_batches)

print('Head importance matrix shape:', head_importance.shape)
print('Min importance: {:.6f}  Max: {:.6f}'.format(
    head_importance.min().item(), head_importance.max().item()))

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(
    head_importance.numpy(),
    annot=True, fmt='.3f', cmap='YlOrRd',
    xticklabels=['H{}'.format(i) for i in range(head_importance.shape[1])],
    yticklabels=['L{}'.format(i) for i in range(head_importance.shape[0])]
)
plt.title('Attention Head Importance Scores (Taylor proxy)')
plt.xlabel('Head')
plt.ylabel('Layer')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'head_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

### 3b – Prune least-important heads

We mask attention heads by zeroing the corresponding output projection weights.
This is *soft pruning* — the architecture is unchanged but selected heads become no-ops.
In production you would reshape weight matrices to fully remove the head dimensions.

In [ ]:
def prune_heads_by_importance(model, importance, prune_ratio=0.3):
    """
    Zero out the output projection weights of the least-important heads.
    prune_ratio: fraction of heads to prune across all layers.
    """
    n_layers, n_heads = importance.shape
    head_dim = model.config.hidden_size // n_heads
    total_heads = n_layers * n_heads
    n_to_prune  = int(total_heads * prune_ratio)

    # Flatten importance and find the lowest-scoring (layer, head) pairs
    flat_imp = importance.view(-1)
    _, sorted_idx = flat_imp.sort()
    pruned_idx = sorted_idx[:n_to_prune]

    pruned_heads = {}
    for flat_i in pruned_idx.tolist():
        li = flat_i // n_heads
        hi = flat_i %  n_heads
        pruned_heads.setdefault(li, []).append(hi)

    with torch.no_grad():
        for li, heads in pruned_heads.items():
            layer = model.bert.encoder.layer[li]
            out_w = layer.attention.output.dense.weight  # (hidden, hidden)
            for hi in heads:
                start = hi * head_dim
                end   = start + head_dim
                out_w[:, start:end] = 0.0

    return model, pruned_heads, n_to_prune


PRUNE_RATIO = 0.30  # prune 30% of heads
model_head_pruned, pruned_heads, n_pruned = prune_heads_by_importance(
    model_head, head_importance, prune_ratio=PRUNE_RATIO
)

print('Pruned {}/{} heads ({:.0f}%)'.format(
    n_pruned, model_head.config.num_hidden_layers * model_head.config.num_attention_heads,
    PRUNE_RATIO * 100
))
print('Pruned heads by layer:', {k: v for k, v in sorted(pruned_heads.items())})

In [ ]:
print('Evaluating head-pruned model ...')
result_head = benchmark_model(
    model_head_pruned, tokenizer, test_texts, test_labels,
    label='Head Pruned (30%)'
)
print('Accuracy  : {:.2f}%  (baseline {:.2f}%)'.format(
    result_head['accuracy'] * 100, baseline['accuracy'] * 100))
print('F1        : {:.4f}  (baseline {:.4f})'.format(
    result_head['f1_weighted'], baseline['f1_weighted']))
print('Speed     : {}ms/sample  (baseline {}ms/sample)'.format(
    result_head['ms_per_sample'], baseline['ms_per_sample']))

## 4 – Strategy B: Layer Dropping (Depth)

### 4a – Estimate layer importance via activation norms

We measure how much each layer transforms the hidden states. Layers with small mean
output-norm relative to their input add little information and are candidates for removal.

In [ ]:
def compute_layer_importance(model, calibration_batches):
    n_layers = model.config.num_hidden_layers
    layer_norms = torch.zeros(n_layers)
    hooks = []
    activations = {}

    def make_hook(idx):
        def hook(module, inp, out):
            # out is a tuple; first element is the layer output tensor
            hidden = out[0] if isinstance(out, tuple) else out
            activations[idx] = hidden.detach()
        return hook

    for i, layer in enumerate(model.bert.encoder.layer):
        h = layer.register_forward_hook(make_hook(i))
        hooks.append(h)

    model.eval()
    with torch.no_grad():
        for batch in calibration_batches:
            inputs = {k: v for k, v in batch.items() if k != 'labels'}
            model(**inputs)
            for idx in range(n_layers):
                layer_norms[idx] += activations[idx].norm(dim=-1).mean().item()

    for h in hooks:
        h.remove()

    return layer_norms / len(calibration_batches)


model_layer = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)

print('Computing layer importance scores ...')
layer_importance = compute_layer_importance(model_layer, cal_batches)

print('Layer importance (activation norm):')
for i, score in enumerate(layer_importance.tolist()):
    print('  Layer {:2d}: {:.4f}'.format(i, score))

In [ ]:
plt.figure(figsize=(10, 4))
bars = plt.bar(range(len(layer_importance)), layer_importance.numpy(), color='steelblue')
plt.xlabel('Layer index')
plt.ylabel('Mean activation norm')
plt.title('Layer Importance (higher = more important)')
plt.xticks(range(len(layer_importance)))
plt.axhline(layer_importance.mean().item(), color='red', linestyle='--',
            label='Mean = {:.2f}'.format(layer_importance.mean().item()))
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'layer_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

### 4b – Drop the least-important layers

In [ ]:
def drop_layers(model, layers_to_drop):
    """
    Remove specific encoder layers by index.
    Returns a new model with the updated encoder.
    """
    kept = [
        layer for i, layer in enumerate(model.bert.encoder.layer)
        if i not in layers_to_drop
    ]
    model.bert.encoder.layer = nn.ModuleList(kept)
    model.config.num_hidden_layers = len(kept)
    return model


# Drop the 3 layers with the lowest activation norm
N_DROP = 3
_, drop_idx = layer_importance.topk(N_DROP, largest=False)
layers_to_drop = sorted(drop_idx.tolist())
print('Dropping layers:', layers_to_drop)

model_layer_pruned = drop_layers(model_layer, layers_to_drop)
print('Remaining layers:', model_layer_pruned.config.num_hidden_layers)

In [ ]:
print('Evaluating layer-dropped model ...')
result_layer = benchmark_model(
    model_layer_pruned, tokenizer, test_texts, test_labels,
    label='Layer Dropped (3 layers)'
)
print('Accuracy  : {:.2f}%  (baseline {:.2f}%)'.format(
    result_layer['accuracy'] * 100, baseline['accuracy'] * 100))
print('F1        : {:.4f}  (baseline {:.4f})'.format(
    result_layer['f1_weighted'], baseline['f1_weighted']))
print('Params    : {}M  (baseline {}M)'.format(
    result_layer['params_M'], baseline['params_M']))
print('Size      : {}MB  (baseline {}MB)'.format(
    result_layer['size_mb'], baseline['size_mb']))

## 5 – Combined pruning: head + layer

We combine both strategies on a fresh model copy to get the maximum compression.

In [ ]:
model_combined = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)

# Recompute importance on the combined model
cal_batches2 = get_calibration_batches(cal_texts, cal_labels, tokenizer)
hi2 = compute_head_importance(model_combined, cal_batches2)
li2 = compute_layer_importance(model_combined, cal_batches2)

# Head pruning first (20%)
model_combined, _, _ = prune_heads_by_importance(model_combined, hi2, prune_ratio=0.20)

# Then drop 2 layers
_, drop_idx2 = li2.topk(2, largest=False)
model_combined = drop_layers(model_combined, sorted(drop_idx2.tolist()))

print('Combined model layers:', model_combined.config.num_hidden_layers)

print('Evaluating combined pruned model ...')
result_combined = benchmark_model(
    model_combined, tokenizer, test_texts, test_labels,
    label='Combined (head 20% + 2 layers)'
)
print('Accuracy  : {:.2f}%'.format(result_combined['accuracy'] * 100))
print('F1        : {:.4f}'.format(result_combined['f1_weighted']))
print('Params    : {}M'.format(result_combined['params_M']))
print('Size      : {}MB'.format(result_combined['size_mb']))

## 6 – Comparison

In [ ]:
all_results = [baseline, result_head, result_layer, result_combined]
labels_r    = [r['label'] for r in all_results]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

metrics = [
    ('accuracy',      'Accuracy',           True),
    ('f1_weighted',   'Weighted F1',        True),
    ('size_mb',       'Model Size (MB)',    False),
    ('ms_per_sample', 'Inference (ms/sample)', False),
]

colors = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

for ax, (metric, title, higher_better) in zip(axes.flat, metrics):
    values = [r[metric] for r in all_results]
    bars = ax.bar(range(len(labels_r)), values, color=colors)
    ax.set_xticks(range(len(labels_r)))
    ax.set_xticklabels(labels_r, rotation=15, ha='right', fontsize=8)
    ax.set_title(title)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01, str(val),
                ha='center', va='bottom', fontsize=8)
    if higher_better:
        ax.axhline(values[0], color='red', linestyle='--', linewidth=0.8, label='Baseline')
        ax.legend(fontsize=7)

plt.suptitle('Pruning Strategies – Performance vs Efficiency', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'pruning_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7 – Save the best pruned model

We save the layer-dropped model as the student for Knowledge Distillation (Notebook 3),
since depth pruning gives the most meaningful parameter reduction.

In [ ]:
STUDENT_DIR = os.path.join(BASE_DIR, 'student_model')
if os.path.exists(STUDENT_DIR): shutil.rmtree(STUDENT_DIR)

# Reload a fresh layer-dropped model to save cleanly
model_student_save = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)
li_save = compute_layer_importance(model_student_save, cal_batches2)
_, drop_save = li_save.topk(N_DROP, largest=False)
model_student_save = drop_layers(model_student_save, sorted(drop_save.tolist()))

model_student_save.save_pretrained(STUDENT_DIR)
tokenizer.save_pretrained(STUDENT_DIR)

# Save pruning results for the final comparison chart
pruning_results = {
    'head_pruned':  {k: v for k, v in result_head.items()     if k != 'predictions'},
    'layer_dropped': {k: v for k, v in result_layer.items()   if k != 'predictions'},
    'combined':     {k: v for k, v in result_combined.items() if k != 'predictions'},
    'layers_dropped': layers_to_drop,
    'n_layers_remaining': N_DROP,
}
with open(os.path.join(BASE_DIR, 'pruning_results.json'), 'w') as f:
    json.dump(pruning_results, f, indent=2)

print('Saved student model to:', STUDENT_DIR)
print('Saved', os.path.join(BASE_DIR, 'pruning_results.json'))

## Summary

| Model | Accuracy | F1 | Size (MB) | Speed (ms) |
|---|---|---|---|---|
| Baseline | — | — | — | — |
| Head pruned 30% | shows above | | | |
| Layer dropped 3 | shows above | | | |
| Combined | shows above | | | |

Pruning reduces size and latency but at the cost of some accuracy.  
**Next step →** [P3_3_Knowledge_Distillation.ipynb](P3_3_Knowledge_Distillation.ipynb)
uses the original FinBERT as a **teacher** to recover the lost performance in the pruned
student model.